# Fase 3: Núcleo Algorítmico y Programación Orientada a Objetos

## Repositorio y procedencia de los datos

### Repositorio del proyecto

https://github.com/DanielCas7illo/U-Andres-Bello

### Fuente del dataset

https://cotera.co/datasets/library/y-combinator-companies

En esta fase se reorganiza el proceso de limpieza desarrollado en F2 mediante Programación Orientada a Objetos, manteniendo las mismas transformaciones realizadas sobre el conjunto de datos de empresas de Y Combinator.

In [ ]:
from pathlib import Path

print(Path.cwd())

In [ ]:
from pathlib import Path
import pandas as pd

RAIZ = Path.cwd().parent

RUTA_DATASET = (
    RAIZ /
    "data" /
    "y-combinator-companies.csv"
)

print(RUTA_DATASET)

df_original = pd.read_csv(RUTA_DATASET)

print(df_original.shape)

# Selección de variables
Se utilizan las mismas variables definidas durante la Fase 2 para mantener la consistencia del análisis.

In [ ]:
variables_analisis = [
    "name",
    "is_operating",
    "is_hiring",
    "nonprofit",
    "status",
    "is_ai",
    "team_size",
    "industry",
    "hq_country",
    "years_since_batch"
]

df = df_original[variables_analisis].copy()

df.head()

# Programación Orientada a Objetos
Durante la Fase 2 las transformaciones fueron implementadas mediante funciones.

En esta fase se reorganiza el proceso utilizando una clase, permitiendo encapsular las operaciones de limpieza y reutilizar el código de manera más eficiente.

# Limpieza de valores nulos y Dato Sucio Ycombinator

In [ ]:
#Se almacena una copia del DF original para evitar modificaciones accidentales
class PreprocesadorStartups:

    def __init__(self, datos):

        self.df = datos.copy()
#Se elimina de las filas la empresa Ycombinator, dado que son los dueños del dataset,
# La idea es que no ensucie el conjunto de datos
    def eliminar_y_combinator(self):

        filas_antes = len(self.df)

        self.df = self.df[
            self.df["name"] != "Y Combinator"
        ]

        filas_despues = len(self.df)

        print(
            f"Registros eliminados: {filas_antes - filas_despues}"
        )

        return self
#Se cambia tamaño de equipo por la mediana
    def imputar_team_size(self):

        mediana = self.df["team_size"].median()

        nulos = self.df["team_size"].isna().sum()

        self.df["team_size"] = (
            self.df["team_size"]
            .fillna(mediana)
        )

        print(
            f"Nulos reemplazados: {nulos}"
        )

        print(
            f"Mediana utilizada: {mediana}"
        )

        return self
#Se cambian valores vacíos de Países y se cambian por desconocidos
    def imputar_pais(self):

        nulos = self.df["hq_country"].isna().sum()

        self.df["hq_country"] = (
            self.df["hq_country"]
            .fillna("Desconocido")
        )

        print(
            f"Paises reemplazados: {nulos}"
        )

        return self
#en variable years_since_batch(años desde que la startup partició en programa Ycombinator), tenía valor negativo
#Se cambia por cero, era negativo porque la startup fue seleccionada pero empieza en 2027
    def corregir_years(self):

        negativos = (
            self.df["years_since_batch"] < 0
        ).sum()

        self.df.loc[
            self.df["years_since_batch"] < 0,
            "years_since_batch"
        ] = 0

        print(
            f"Valores negativos corregidos: {negativos}"
        )

        return self

    def procesar(self):

        (
            self
            .eliminar_y_combinator()
            .imputar_team_size()
            .imputar_pais()
            .corregir_years()
        )

        return self.df

In [ ]:
preprocesador = PreprocesadorStartups(df)

df_procesado = preprocesador.procesar()

df_procesado.head()

# Validación Técnica

A continuación se verifica que las transformaciones fueron aplicadas correctamente

In [ ]:
assert (
    df_procesado["team_size"].isna().sum() == 0
)

assert (
    df_procesado["hq_country"].isna().sum() == 0
)

assert (
    (df_procesado["years_since_batch"] < 0).sum() == 0
)

assert (
    "Y Combinator"
    not in df_procesado["name"].values
)

print("Validaciones superadas")